# Лекция: Факторный анализ в Python

**Дисциплина:** Введение в анализ больших данных  
**Задание 13** (адаптация с языка R на Python)

## Краткая теория

**Факторный анализ** — метод снижения размерности: много наблюдаемых переменных объясняются меньшим числом **латентных факторов**.

- **Факторные нагрузки (factor loadings)** — корреляции переменных с факторами
- **Общность (communality)** — доля дисперсии переменной, объяснённая факторами
- **Собственные значения** — вклад каждого фактора
- **Вращение (rotation)** — varimax / promax для интерпретируемости

В R: `factanal()`, `fa.parallel()` (psych)  
В Python: **sklearn FactorAnalysis**, PCA scree, varimax.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")
np.random.seed(42)
print("Библиотеки загружены")


---
## 1. Данные (опрос / шкалы)

В задании: **FactorAn.csv** — 100 студентов, 10 вопросов об удовлетворённости обучением.
Ниже — демонстрационный набор с 2–3 латентными факторами.


In [ ]:
n = 100
F1 = np.random.normal(0, 1, n)
F2 = np.random.normal(0, 1, n)
F3 = np.random.normal(0, 1, n)

items = pd.DataFrame({
    "Q1_lectures":  0.8*F1 + 0.1*F2 + np.random.normal(0, 0.4, n),
    "Q2_seminars":  0.7*F1 + 0.2*F2 + np.random.normal(0, 0.4, n),
    "Q3_teachers":  0.75*F1 + np.random.normal(0, 0.4, n),
    "Q4_materials": 0.65*F1 + 0.15*F2 + np.random.normal(0, 0.5, n),
    "Q5_labs":      0.2*F1 + 0.7*F2 + np.random.normal(0, 0.4, n),
    "Q6_library":   0.1*F1 + 0.75*F2 + np.random.normal(0, 0.4, n),
    "Q7_campus":    0.8*F2 + np.random.normal(0, 0.45, n),
    "Q8_wifi":      0.6*F2 + 0.2*F3 + np.random.normal(0, 0.5, n),
    "Q9_clubs":     0.1*F2 + 0.75*F3 + np.random.normal(0, 0.4, n),
    "Q10_events":   0.8*F3 + np.random.normal(0, 0.4, n),
})
items = (3 + items).clip(1, 5)
print(items.describe().round(2).T)


### Корреляционная матрица (R-matrix)


In [ ]:
corr = items.corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlBu_r", center=0,
            vmin=-1, vmax=1, square=True)
plt.title("Корреляционная матрица пунктов опроса")
plt.tight_layout()
plt.show()


---
## 2. Число факторов: scree plot и параллельный анализ

В R: `fa.parallel` из пакета **psych**


In [ ]:
eigvals = np.linalg.eigvalsh(corr.values)[::-1]
print("Собственные значения:", np.round(eigvals, 3))

n_perm = 50
rand_eigs = []
for _ in range(n_perm):
    R = np.random.normal(size=items.shape)
    R = StandardScaler().fit_transform(R)
    C = np.corrcoef(R.T)
    rand_eigs.append(np.linalg.eigvalsh(C)[::-1])
rand_eigs = np.mean(rand_eigs, axis=0)

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(eigvals)+1), eigvals, "o-", label="Данные")
plt.plot(range(1, len(rand_eigs)+1), rand_eigs, "s--", label="Случайные (parallel)")
plt.axhline(1, color="gray", ls=":", label="Kaiser (eigen=1)")
plt.xlabel("Номер фактора")
plt.ylabel("Собственное значение")
plt.title("Scree plot + parallel analysis")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

n_factors_pa = int(np.sum(eigvals > rand_eigs))
print(f"По parallel analysis рекомендуемое число факторов: {n_factors_pa}")


---
## 3. Факторный анализ (2 и 3 фактора)

В R: `factanal(x, factors = k, rotation = "varimax")`


In [ ]:
def varimax(Phi, gamma=1.0, q=20, tol=1e-6):
    p, k = Phi.shape
    R = np.eye(k)
    d = 0
    for _ in range(q):
        d_old = d
        Lambda = Phi @ R
        u, s, vh = np.linalg.svd(
            Phi.T @ (Lambda**3 - (gamma / p) * Lambda @ np.diag(np.diag(Lambda.T @ Lambda)))
        )
        R = u @ vh
        d = np.sum(s)
        if d_old != 0 and d / d_old < 1 + tol:
            break
    return Phi @ R

def fit_fa(data, n_factors, rotation="varimax"):
    fa = FactorAnalysis(n_components=n_factors, random_state=42, max_iter=1000)
    fa.fit(data)
    loadings = fa.components_.T
    if rotation == "varimax":
        loadings = varimax(loadings)
    load_df = pd.DataFrame(
        loadings, index=data.columns,
        columns=[f"F{i+1}" for i in range(n_factors)]
    )
    load_df["communality"] = np.sum(loadings**2, axis=1)
    return load_df, fa

print("=== 2 фактора ===")
load2, fa2 = fit_fa(items, 2)
print(load2.round(3))


In [ ]:
print("=== 3 фактора ===")
load3, fa3 = fit_fa(items, 3)
print(load3.round(3))


In [ ]:
plot_df = load3.drop(columns=["communality"])
plt.figure(figsize=(6, 6))
sns.heatmap(plot_df, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Факторные нагрузки (3 фактора, varimax)")
plt.tight_layout()
plt.show()


### Интерпретация нагрузок

- |нагрузка| > 0.4–0.5 — переменная «принадлежит» фактору
- Высокая **communality** — переменная хорошо объясняется факторами
- Дайте факторам содержательные имена


---
## 4. Задание: FactorAn.csv

```python
df = pd.read_csv("FactorAn.csv")
X = df.select_dtypes(include=[np.number])
sns.heatmap(X.corr(), annot=True, fmt=".2f", cmap="RdYlBu_r", center=0)
eigvals = np.linalg.eigvalsh(X.corr())[::-1]
plt.plot(range(1, len(eigvals)+1), eigvals, "o-")
for k in [2, 3]:
    loads, model = fit_fa(X, k)
    print(loads.round(3))
```


In [ ]:
def kmo_simple(corr):
    corr = np.asarray(corr)
    inv = np.linalg.inv(corr)
    pcorr = -inv / np.sqrt(np.outer(np.diag(inv), np.diag(inv)))
    np.fill_diagonal(pcorr, 0)
    np.fill_diagonal(corr, 0)
    r2 = (corr**2).sum()
    p2 = (pcorr**2).sum()
    return r2 / (r2 + p2)

kmo = kmo_simple(corr.values.copy())
print(f"KMO ≈ {kmo:.3f}  (желательно > 0.6)")
print("Если KMO низкий — FA может быть неуместен.")


---
## Шпаргалка: R → Python

| Задача в R | Python |
|------------|--------|
| `cor(x)` | `df.corr()` |
| `fa.parallel` (psych) | scree + random eigenvalues |
| `factanal(x, factors=k, rotation="varimax")` | `FactorAnalysis` + varimax |
| `loadings(fit)` | `components_.T` после rotation |
| `fit$uniquenesses` | `1 - communality` |

---
## Рекомендации

1. Перед FA проверьте KMO.
2. Число факторов: parallel analysis + интерпретируемость.
3. Файл **FactorAn.csv** подставьте локально.
4. Опционально: `pip install factor_analyzer`

**Удачи с выполнением Задания 13!**
